# v4 dash-cam pothole model — beat 0.182 on the Detroit exam (Ayaan's brief, chat 01:04 UTC)

**Bar:** shipped dash-cam model = v3+v1 weight blend 70/30, Detroit exam box F1 **0.182** (v3 0.157, v1 0.103).
**v4 data** (`pipeline/autolabel/build_v4_data.py`): Detroit train + road-band full-res tiles + v3's mistakes ×3
(mostly tar snakes / lane edges) + pasted small potholes, ~50% "nothing here". Split by video, test never touched.
**+ RDD2022 United States** (Ayaan, chat 01:14 UTC): US road-damage photos from car cameras via the HF mirror
`dronefreak/RDD2022` (CC BY-SA 4.0). Pothole = class 3 (D40) → our class 0; crack-only photos kept as hard negatives
(v3's #1 false box is tar snakes / sealed cracks). Capped at 3× the Detroit image count so Detroit isn't swamped.
Validation = Detroit only, so `best.pt` is chosen for Detroit.
**Train** from v3 at 640. **Then** blend v4 with v1 (70/30 and 85/15) and score everything — including the shipped
blend as a sanity baseline — on the same 215-frame exam.

**Setup:** A100 (or L4) · `Runtime → Run all` · ~40–50 min. Last cell prints `report.md` and saves the best
ONNX to `still-there-potholes/handoff_v4/`. **Ship rule:** best box F1 wins, must beat 0.182.

## 1 · GPU, Drive, repo, v3 weights

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU"
!pip -q install ultralytics onnx onnxruntime onnxslim huggingface_hub
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, glob
DRIVE_DIR = '/content/drive/MyDrive/still-there-potholes'
if os.path.isdir('/content/repo/.git'):
    !git -C /content/repo pull -q
else:
    !git clone -q https://github.com/ayaanamir/RoadMapWayneAWSHackathon /content/repo
V3 = next(p for p in (f'{DRIVE_DIR}/handoff_detroit/best.pt', f'{DRIVE_DIR}/runs/y11n_dashcam_detroit/weights/best.pt') if os.path.exists(p))
shutil.copy(V3, '/content/v3_best.pt')
V1_ONNX = '/content/repo/pipeline/train/results/v1_phone/pothole.onnx'
SHIPPED = '/content/repo/apps/live/model/pothole-dashcam.onnx'   # v3+v1 blend, exam 0.182
for p in ('/content/v3_best.pt', V1_ONNX, SHIPPED): print(os.path.exists(p), p)

## 2a · RDD2022 United States → one class `pothole` (+ crack-only hard negatives)

In [ ]:
RDD_CODE = True  # marker for the offline test
import random
from huggingface_hub import snapshot_download
def build_rdd_us(src='/content/rdd', out='/content/rdd_us', seed=0):
    random.seed(seed)
    for d in ('images', 'labels'): os.makedirs(f'{out}/{d}/train', exist_ok=True)
    pos, neg, crack_exam = [], [], []
    for lab in sorted(glob.glob(f'{src}/data/labels/*/*/United_States_*.txt')):
        if '/test/' in lab.replace('\\', '/'):                           # RDD test split → held out for the crack check
            rows = [l.split() for l in open(lab) if l.strip()]
            if rows and not any(r[0] == '3' for r in rows):                 # crack-only, no pothole
                img = lab.replace('/labels/', '/images/')[:-4] + '.jpg'
                if os.path.exists(img): crack_exam.append(img)
            continue
        rows = [l.split() for l in open(lab) if l.strip()]
        pot = [r for r in rows if r[0] == '3']                                    # D40 = pothole
        (pos if pot else neg).append((lab, pot, bool(rows)))
    cracks = [n for n in neg if n[2]]                                             # crack-only = hard negatives
    negs = random.sample(cracks, min(len(cracks), len(pos)))                      # ≈1:1 with pothole photos
    kept = []
    for lab, pot, _ in pos + negs:
        img = lab.replace('/labels/', '/images/')[:-4] + '.jpg'
        if not os.path.exists(img): continue
        name = 'rddus_' + os.path.basename(img)[:-4]
        shutil.copy(img, f'{out}/images/train/{name}.jpg')
        open(f'{out}/labels/train/{name}.txt', 'w').write(''.join('0 ' + ' '.join(r[1:5]) + '\n' for r in pot))
        kept.append(name)
    os.makedirs('/content/crack_exam', exist_ok=True)
    for img in crack_exam: shutil.copy(img, '/content/crack_exam/')
    print('RDD US (train+val) pothole images:', len(pos), '| pothole boxes:', sum(len(p) for _, p, _ in pos),
          '| crack-only negatives kept:', len(negs), '| written:', len(kept),
          '| held-out crack-only test images (crack check):', len(crack_exam))
    return kept

if not glob.glob('/content/rdd_us/images/train/*.jpg'):
    snapshot_download('dronefreak/RDD2022', repo_type='dataset', local_dir='/content/rdd',
                      allow_patterns=['data/images/*/*/United_States_*', 'data/labels/*/*/United_States_*.txt'], max_workers=16)
    RDD_KEPT = build_rdd_us()
else:
    RDD_KEPT = [os.path.basename(p)[:-4] for p in glob.glob('/content/rdd_us/images/train/*.jpg')]
    print('RDD US already built:', len(RDD_KEPT))

## 2b · Detroit data → v4 training set

In [ ]:
!python3 /content/repo/pipeline/autolabel/pull_reviewed.py --api https://wayneawshackathon.vercel.app --out /content/detroit
!python3 /content/repo/pipeline/autolabel/build_v4_data.py --images /content/detroit/train/images --labels /content/detroit/train/labels --model /content/v3_best.pt --conf 0.35 --out /content/ds_v4
print(open('/content/ds_v4/data.yaml').read())
for s in ('train', 'val'):
    d = f'/content/ds_v4/images/{s}'
    if os.path.isdir(d):
        labs = glob.glob(f'/content/ds_v4/labels/{s}/*.txt')
        print(s, len(os.listdir(d)), 'images,', sum(os.path.getsize(l) == 0 for l in labs), 'empty labels')

## 2c · One yaml: train on Detroit v4 + RDD US (capped at 3× Detroit), validate on Detroit only

In [ ]:
import yaml
n_det = len(glob.glob('/content/ds_v4/images/train/*'))
rdd_imgs = sorted(glob.glob('/content/rdd_us/images/train/*.jpg'))
cap = 3 * n_det
if len(rdd_imgs) > cap:                                    # keep Detroit from being swamped
    random.seed(1)
    for p in random.sample(rdd_imgs, len(rdd_imgs) - cap):
        os.remove(p); os.remove(p.replace('/images/', '/labels/')[:-4] + '.txt')
    for c in glob.glob('/content/rdd_us/labels/*.cache'): os.remove(c)
n_rdd = len(glob.glob('/content/rdd_us/images/train/*.jpg'))
yaml.safe_dump({'train': ['/content/ds_v4/images/train', '/content/rdd_us/images/train'],
                'val': '/content/ds_v4/images/val', 'nc': 1, 'names': ['pothole']}, open('/content/v4_rdd.yaml', 'w'))
print(f'train = Detroit v4 {n_det} + RDD US {n_rdd} images (cap {cap}); val = Detroit only')

## 3 · Train v4 (+RDD US) from v3 (≈25–40 min on A100) — resumes after a disconnect

In [ ]:
from ultralytics import YOLO
RUN = f'{DRIVE_DIR}/runs/v4_rdd'
last, done = f'{RUN}/weights/last.pt', f'{RUN}/weights/TRAINING_DONE'
if os.path.exists(done):
    print('Already trained — skipping. Delete', done, 'to train again.')
elif os.path.exists(last):
    print('Resuming', last); YOLO(last).train(resume=True)
else:
    YOLO('/content/v3_best.pt').train(data='/content/v4_rdd.yaml', imgsz=640, epochs=40, patience=12,
        lr0=0.002, warmup_epochs=1, hsv_v=0.5, hsv_s=0.7, close_mosaic=10, batch=-1, cache='ram',
        workers=min(8, os.cpu_count() or 2), project=f'{DRIVE_DIR}/runs', name='v4_rdd', exist_ok=True)
open(done, 'w').close()
V4_PT = f'{RUN}/weights/best.pt'

## 4 · Export v4 + blend with v1 (70/30, 85/15) — the trick that took v3 0.157 → 0.182

In [ ]:
import onnx, numpy as np
from onnx import numpy_helper
V4 = YOLO(V4_PT).export(format='onnx', imgsz=640)
def blend(a_path, b_path, a, out):
    A = onnx.load(a_path); B = {i.name: numpy_helper.to_array(i) for i in onnx.load(b_path).graph.initializer}
    n = 0
    for k, init in enumerate(A.graph.initializer):
        w = numpy_helper.to_array(init)
        if w.dtype == np.float32 and init.name in B and B[init.name].shape == w.shape:
            A.graph.initializer[k].CopyFrom(numpy_helper.from_array((a*w + (1-a)*B[init.name]).astype(np.float32), init.name)); n += 1
    onnx.save(A, out); print(f'{os.path.basename(out)}: blended {n}/{len(A.graph.initializer)} tensors')
for a in (0.7, 0.85): blend(V4, V1_ONNX, a, f'/content/v4v1_{int(a*100)}.onnx')

## 5 · The Detroit exam — v4, both blends, and the shipped blend (0.182) as baseline

In [ ]:
CANDS = {'v4': V4, 'v4v1_70': '/content/v4v1_70.onnx', 'v4v1_85': '/content/v4v1_85.onnx', 'shipped_v3v1_70': SHIPPED}
named = {}
for name, p in CANDS.items():                      # distinct file names so the report rows are readable
    dst = f'/content/exam_models/{name}{os.path.splitext(p)[1]}'; os.makedirs('/content/exam_models', exist_ok=True)
    shutil.copy(p, dst); named[name] = dst
args = ' '.join(f'--model {p}' for p in named.values())
!python3 /content/repo/pipeline/autolabel/eval_models.py --images /content/detroit/test/images --labels /content/detroit/test/labels --out /content/eval_v4 --modes 640 --device 0 {args}
REPORT = open('/content/eval_v4/report.md').read(); print(REPORT)
os.makedirs(f'{DRIVE_DIR}/handoff_v4', exist_ok=True)
open(f'{DRIVE_DIR}/handoff_v4/v4_exam_report.md', 'w').write(REPORT)

## 5b · Crack check — does it still call cracks potholes?

Held-out RDD US **crack-only** test photos (never trained on). Reports the % of photos where each model draws a
pothole box at conf ≥ 0.30 (the live threshold). Lower = better at telling cracks from potholes. Pick the ship
candidate on Detroit box F1 first; use this to break ties and to judge the tar-snake problem.

In [ ]:
CRACKS = sorted(glob.glob('/content/crack_exam/*.jpg'))
print(len(CRACKS), 'crack-only photos')
lines = ['| model | crack photos with a false pothole box (conf ≥ 0.30) |', '|---|---|']
for name, p in named.items():
    m = YOLO(p, task='detect'); hit = 0
    for i in range(0, len(CRACKS), 32):
        for r in m.predict(CRACKS[i:i+32], conf=0.30, imgsz=640, verbose=False): hit += len(r.boxes) > 0
    lines.append(f'| {name} | {hit}/{len(CRACKS)} = {hit/max(1,len(CRACKS)):.0%} |')
CRACK_REPORT = '\n'.join(lines); print(CRACK_REPORT)
open(f'{DRIVE_DIR}/handoff_v4/v4_exam_report.md', 'a').write('\n\n## Crack check (held-out RDD US crack-only photos)\n' + CRACK_REPORT + '\n')

## 6 · Save the candidates to Drive (`handoff_v4/`), then disconnect

Ship only the winner, and only if it beats **0.182**: post the report + its Drive link in the agent chat;
Ayaan puts it at `apps/live/model/pothole-dashcam.onnx` (one PR, one deploy). Don't commit .pt/.onnx yourself.

In [ ]:
for name in ('v4', 'v4v1_70', 'v4v1_85'):
    shutil.copy(named[name], f'{DRIVE_DIR}/handoff_v4/{name}.onnx')
shutil.copy(V4_PT, f'{DRIVE_DIR}/handoff_v4/v4_best.pt')
for f in sorted(os.listdir(f'{DRIVE_DIR}/handoff_v4')): print(f'{f:22s} {os.path.getsize(f"{DRIVE_DIR}/handoff_v4/{f}")/1e6:.1f} MB')
print('\nDONE — Runtime → Disconnect and delete runtime to stop burning units.')